# 3.5 — DataFrame Joins

**Chapter 3, section 3.9.1** (*Writing a Join*), and Exercise 5.

**The question this notebook answers:** how is a join written, what does each `how` actually
return, and what do you do about the column that appears twice in the result?

The last of those is the mundane difficulty that costs beginners the most time. Join two tables
that share a column name beyond the key and the result carries two columns of that name; any
reference to one of them is ambiguous, and Spark raises rather than guessing. The notebook
reproduces that failure deliberately and then shows the two ways out.

*How* Spark executes a join — broadcast hash, sort-merge, shuffle hash — is a separate question,
taken up in [3.6](03.06%20Catalyst%20AQE%20and%20Skew.ipynb).

Runs on a laptop in well under a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

DATA = os.environ.get("CS777_DATA", "../data")          # -> code/data/
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-3.5")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")   # keep printed output clean
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version)

Spark 4.2.0


## The two tables

Five employees and three departments. Note two things that will matter: `Eve` is in HR, so
every department has at least one employee except none — and there is a fourth department,
`Legal`, with no employees at all. That unmatched row on each side is what makes the difference
between the four join types visible.

In [2]:
from pyspark.sql.types import StringType, StructField, IntegerType, StructType
# Create a list of tuples with sample employee data
employee_data = [
    ("1", "Alice", 30, "Engineering"),
    ("2", "Bob", 25, "Engineering"),
    ("3", "Charlie", 35, "HR"),
    ("4", "David", 28, "Finance"),
    ("5", "Eve", 22, "HR"),
    ("6", "Frank", 41, "Research"),     # an employee in no listed department
]

# Define the schema for the employee DataFrame
employee_schema = StructType([
    StructField("EmployeeID", StringType(), True),
    StructField("Name", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Department", StringType(), True)
])

# Create the employee DataFrame
employee_df = spark.createDataFrame(employee_data, employee_schema)

# Create a list of tuples with department data
department_data = [
    ("Engineering", "New York"),
    ("HR",          "San Francisco"),
    ("Finance",     "Los Angeles"),
    ("Legal",       "Boston"),          # a department with nobody in it
]

# Define the schema for the department DataFrame
department_schema = StructType([
    StructField("Department", StringType(), True),
    StructField("Location", StringType(), True)
])
department_df = spark.createDataFrame(department_data, department_schema)

employee_df.show()
department_df.show()

+----------+-------+---+-----------+
|EmployeeID|   Name|Age| Department|
+----------+-------+---+-----------+
|         1|  Alice| 30|Engineering|
|         2|    Bob| 25|Engineering|
|         3|Charlie| 35|         HR|
|         4|  David| 28|    Finance|
|         5|    Eve| 22|         HR|
|         6|  Frank| 41|   Research|
+----------+-------+---+-----------+

+-----------+-------------+
| Department|     Location|
+-----------+-------------+
|Engineering|     New York|
|         HR|San Francisco|
|    Finance|  Los Angeles|
|      Legal|       Boston|
+-----------+-------------+



## The four join types

The `how` argument selects among the same four variants met earlier for RDDs. `inner`, the
default, keeps only the keys present on both sides. `left` and `right` keep every row of the
named side, filling the other side's columns with nulls where no match exists. `full` keeps
every key from either side.

Frank (in `Research`, which is not a listed department) and Legal (a department with no
employees) are the rows that separate them.

In [3]:
for how in ["inner", "left", "right", "full"]:
    joined = employee_df.join(department_df, employee_df.Department == department_df.Department, how)
    print(f"how={how!r}: {joined.count()} rows")
    joined.orderBy("EmployeeID").show()

how='inner': 5 rows
+----------+-------+---+-----------+-----------+-------------+
|EmployeeID|   Name|Age| Department| Department|     Location|
+----------+-------+---+-----------+-----------+-------------+
|         1|  Alice| 30|Engineering|Engineering|     New York|
|         2|    Bob| 25|Engineering|Engineering|     New York|
|         3|Charlie| 35|         HR|         HR|San Francisco|
|         4|  David| 28|    Finance|    Finance|  Los Angeles|
|         5|    Eve| 22|         HR|         HR|San Francisco|
+----------+-------+---+-----------+-----------+-------------+



how='left': 6 rows
+----------+-------+---+-----------+-----------+-------------+
|EmployeeID|   Name|Age| Department| Department|     Location|
+----------+-------+---+-----------+-----------+-------------+
|         1|  Alice| 30|Engineering|Engineering|     New York|
|         2|    Bob| 25|Engineering|Engineering|     New York|
|         3|Charlie| 35|         HR|         HR|San Francisco|
|         4|  David| 28|    Finance|    Finance|  Los Angeles|
|         5|    Eve| 22|         HR|         HR|San Francisco|
|         6|  Frank| 41|   Research|       NULL|         NULL|
+----------+-------+---+-----------+-----------+-------------+



how='right': 6 rows
+----------+-------+----+-----------+-----------+-------------+
|EmployeeID|   Name| Age| Department| Department|     Location|
+----------+-------+----+-----------+-----------+-------------+
|      NULL|   NULL|NULL|       NULL|      Legal|       Boston|
|         1|  Alice|  30|Engineering|Engineering|     New York|
|         2|    Bob|  25|Engineering|Engineering|     New York|
|         3|Charlie|  35|         HR|         HR|San Francisco|
|         4|  David|  28|    Finance|    Finance|  Los Angeles|
|         5|    Eve|  22|         HR|         HR|San Francisco|
+----------+-------+----+-----------+-----------+-------------+



how='full': 7 rows
+----------+-------+----+-----------+-----------+-------------+
|EmployeeID|   Name| Age| Department| Department|     Location|
+----------+-------+----+-----------+-----------+-------------+
|      NULL|   NULL|NULL|       NULL|      Legal|       Boston|
|         1|  Alice|  30|Engineering|Engineering|     New York|
|         2|    Bob|  25|Engineering|Engineering|     New York|
|         3|Charlie|  35|         HR|         HR|San Francisco|
|         4|  David|  28|    Finance|    Finance|  Los Angeles|
|         5|    Eve|  22|         HR|         HR|San Francisco|
|         6|  Frank|  41|   Research|       NULL|         NULL|
+----------+-------+----+-----------+-----------+-------------+



`inner` loses both odd rows; `left` keeps Frank with a null location; `right` keeps Legal with
a null employee; `full` keeps both. Two further variants exist and are worth knowing because
they return only one side's columns: `left_semi` keeps the rows of the left side that *have* a
match (a filter, not a join), and `left_anti` keeps the rows that do not — which is the
idiomatic way to ask "which employees are in no listed department?"

In [4]:
print("left_semi: employees whose department is listed")
employee_df.join(department_df, employee_df.Department == department_df.Department, "left_semi") \
           .orderBy("EmployeeID").show()

print("left_anti: employees whose department is NOT listed")
employee_df.join(department_df, employee_df.Department == department_df.Department, "left_anti") \
           .orderBy("EmployeeID").show()

left_semi: employees whose department is listed
+----------+-------+---+-----------+
|EmployeeID|   Name|Age| Department|
+----------+-------+---+-----------+
|         1|  Alice| 30|Engineering|
|         2|    Bob| 25|Engineering|
|         3|Charlie| 35|         HR|
|         4|  David| 28|    Finance|
|         5|    Eve| 22|         HR|
+----------+-------+---+-----------+

left_anti: employees whose department is NOT listed


+----------+-----+---+----------+
|EmployeeID| Name|Age|Department|
+----------+-----+---+----------+
|         6|Frank| 41|  Research|
+----------+-----+---+----------+



## The duplicated key column

Look at the joins above: `Department` appears **twice** in every result, once from each side.
That is what the boolean-expression form of the condition does — it compares two columns and
keeps both.

In [5]:
emp_dept = employee_df.join(department_df, employee_df.Department == department_df.Department, 'inner')
emp_dept.show()

+----------+-------+---+-----------+-----------+-------------+
|EmployeeID|   Name|Age| Department| Department|     Location|
+----------+-------+---+-----------+-----------+-------------+
|         1|  Alice| 30|Engineering|Engineering|     New York|
|         2|    Bob| 25|Engineering|Engineering|     New York|
|         4|  David| 28|    Finance|    Finance|  Los Angeles|
|         3|Charlie| 35|         HR|         HR|San Francisco|
|         5|    Eve| 22|         HR|         HR|San Francisco|
+----------+-------+---+-----------+-----------+-------------+



In [6]:
print("columns of the joined result:", emp_dept.columns)

columns of the joined result: ['EmployeeID', 'Name', 'Age', 'Department', 'Department', 'Location']


Selecting `Department` from that result is ambiguous, and Spark says so rather than guessing.
This is the exception the notebook exists to reproduce:

In [7]:
# PySpark also logs a full structured record for every DataFrame analysis error.
# It is useful in a job log and unhelpful here, where the failure is deliberate.
import logging
logging.getLogger("DataFrameQueryContextLogger").setLevel(logging.CRITICAL)

# This raises: "Reference `Department` is ambiguous, could be: Department, Department."
try:
    emp_dept.select('Name', 'Department', 'Location').show()
except Exception as e:
    print(type(e).__name__)
    print(str(e).split("\n")[0])

AnalysisException
[AMBIGUOUS_REFERENCE] Reference `Department` is ambiguous, could be: [`Department`, `Department`]. SQLSTATE: 42704


### Remedy 1: name the key instead of comparing two columns

When the key bears the same name on both sides — as it does here — the condition can be given
as that name alone (or a list of names for a compound key). Spark then treats it as a single
join key and the result carries **one** `Department` column. This is the form to reach for
first, and it makes the ambiguity impossible rather than merely survivable.

In [8]:
by_name = employee_df.join(department_df, on="Department", how="inner")
print("columns:", by_name.columns)
by_name.select("Name", "Department", "Location").orderBy("Name").show()

columns: ['Department', 'EmployeeID', 'Name', 'Age', 'Location']
+-------+-----------+-------------+
|   Name| Department|     Location|
+-------+-----------+-------------+
|  Alice|Engineering|     New York|
|    Bob|Engineering|     New York|
|Charlie|         HR|San Francisco|
|  David|    Finance|  Los Angeles|
|    Eve|         HR|San Francisco|
+-------+-----------+-------------+



### Remedy 2: alias each side

When the columns are genuinely distinct — different names on each side, or a column other than
the key that happens to be shared — give each input an alias with `alias` and qualify the
ambiguous column by it, exactly as one disambiguates table names in SQL.

In [9]:
emp_dept = employee_df.alias('emp').join(department_df.alias('dept'), employee_df.Department == department_df.Department, 'inner')
emp_dept.show()

+----------+-------+---+-----------+-----------+-------------+
|EmployeeID|   Name|Age| Department| Department|     Location|
+----------+-------+---+-----------+-----------+-------------+
|         1|  Alice| 30|Engineering|Engineering|     New York|
|         2|    Bob| 25|Engineering|Engineering|     New York|
|         4|  David| 28|    Finance|    Finance|  Los Angeles|
|         3|Charlie| 35|         HR|         HR|San Francisco|
|         5|    Eve| 22|         HR|         HR|San Francisco|
+----------+-------+---+-----------+-----------+-------------+



In [10]:
emp_dept.select('Name', 'dept.Department', 'emp.Department','dept.Location').orderBy('Name').show()

+-------+-----------+-----------+-------------+
|   Name| Department| Department|     Location|
+-------+-----------+-----------+-------------+
|  Alice|Engineering|Engineering|     New York|
|    Bob|Engineering|Engineering|     New York|
|Charlie|         HR|         HR|San Francisco|
|  David|    Finance|    Finance|  Los Angeles|
|    Eve|         HR|         HR|San Francisco|
+-------+-----------+-----------+-------------+



## When the key names differ

If the two sides call the key by different names there is no ambiguity to begin with, and the
condition is written as a boolean column expression. Both key columns survive into the result,
which is usually what you want — and `drop` removes the redundant one when it is not.

In [11]:
staff = employee_df.withColumnRenamed("Department", "dept")
joined = staff.join(department_df, staff.dept == department_df.Department, "inner")
print("columns:", joined.columns)
joined.select("Name", "dept", "Location").orderBy("Name").show()

# and if the duplicate is not wanted:
joined.drop("Department").orderBy("Name").show()

columns: ['EmployeeID', 'Name', 'Age', 'dept', 'Department', 'Location']
+-------+-----------+-------------+
|   Name|       dept|     Location|
+-------+-----------+-------------+
|  Alice|Engineering|     New York|
|    Bob|Engineering|     New York|
|Charlie|         HR|San Francisco|
|  David|    Finance|  Los Angeles|
|    Eve|         HR|San Francisco|
+-------+-----------+-------------+



+----------+-------+---+-----------+-------------+
|EmployeeID|   Name|Age|       dept|     Location|
+----------+-------+---+-----------+-------------+
|         1|  Alice| 30|Engineering|     New York|
|         2|    Bob| 25|Engineering|     New York|
|         3|Charlie| 35|         HR|San Francisco|
|         4|  David| 28|    Finance|  Los Angeles|
|         5|    Eve| 22|         HR|San Francisco|
+----------+-------+---+-----------+-------------+



## Conclusion

Three ways to write the condition, and the choice among them decides what the result's columns
look like:

| Condition | Result | Use when |
|-----------|--------|----------|
| `on="Department"` | **one** key column | the key has the same name on both sides — prefer this |
| `left.Department == right.Department` | **two** key columns, ambiguous by name | you need both, or the names differ |
| `left.dept == right.Department` | two differently named columns | the names already differ |

And the four `how` values in one sentence: `inner` keeps matched keys, `left` and `right` keep
every row of the named side with nulls opposite, `full` keeps everything; `left_semi` and
`left_anti` return only the left side's columns and are filters in disguise.

The ambiguous-reference exception above is worth having seen once deliberately. It is not a bug
in the join; it is Spark refusing to guess which of two identically named columns you meant,
and the fix is always to be explicit — by naming the key, or by naming the side.

Where this goes next: [3.6](03.06%20Catalyst%20AQE%20and%20Skew.ipynb) asks the other question
about a join — not how it is written, but which physical strategy Spark chooses to execute it,
and what happens when one key holds most of the rows.